<center>

# Surface Defect Classification
</center>




<br>

----

<br>



## Description

Surface defect classification is a critical task in various industries, including manufacturing, quality control, and materials science. The goal is to identify and categorize defects on surfaces of manufactured products, which can range from functional to aesthetic issues. Accurate classification of surface defects is essential for ensuring product quality, reducing waste, and improving overall efficiency in production processes. Misclassification can lead to increased costs as rework or scrapping may be necessary. Additionally, customer dissatisfaction can lead to warranty claims, returns, and damage to the brand's reputation [1]. The consequences of inaccurate classification can extend beyond cosmetic, productivity, or reputation issues. In many industries, such as in automotive or aerospace, surface defects mean failure precursors that can lead to catastrophic consequences. For instance, micro-cracks in metal can lead to fatigue failure or delamination in composite materials can to structural failure [2]. Therefore, accurate classification of surface defects plays a vital role for ensuring the safety and long-term reliability of products in these industries.

In the past, methods relied on local features and handcrafted algorithms, such as structural edge, skeleton, or template matching. Thresholding approaches were also widely used, including iterative optimal thresholding as Otsu’s and Kittler’s techniques. However, these methods often become labor-intensive and may fail to achieve the required level of accuracy in complex industrial environments [3]. In recent years, Machine Learning (ML) has become increasingly adopted in industrial inspection tasks due to its ability to learn complex patterns and representative features directly from data, resulting in improved accuracy and efficiency [1] (glaube aber fact check). Surface defect classification/detection has become one of the most important application areas of ML, as almost every manufacturing product has a visible of functional surface that hast to meet certain quality standards (fact check).

Despite the significant advances achieved with classical ML techniques such as Neural Networks (NNs), they do not provide perfect performance. In search for improved performance exploring alternative approaches, such as Quantum Machine Learning (QML), is gaining more and more popularity.
QML aims to leverage the principles of quantum mechanics to enhance the capabilities of traditional ML algorithms. One such approach is the use of Quantum Boltzmann Machines (QBMs), which are quantum analogs of classical Boltzmann Machines (BMs). QBMs can learn complex probability distributions, sometimes more efficiently than their classical counterparts, and, when paired with Quantum Annealing (QA), they can perform the sampling process required for training and inference potentially faster than classical methods [4]. We pair this promising approach with a concept that has helped classical NNs to achieve state-of-the-art performance in such classification tasks: convolution. Convolutional Deep Quantum Boltzmann Machines (CDQBMs) are a variant of QBMs that incorporate convolutional layers, allowing them to capture spatial hierarchies and local features in data, which is particularly beneficial for image classification tasks as required by this use case.

For this showcase, we will explore the application of this QML-approach for surface defects of  hot-rolled steel strip. The Northeastern University (NEU) surface defect database will give as the necessary data to learn how to distinguish between pits and rust on 64x64 grayscale images (more detailed intro of dataset). We will compare the performance of the CDQBM with a classical Convolutional Neural Network (CNN) to evaluate the potential advantages of the quantum approach in this context.


## Background


Before we dive into the implementation of the CDQBM, a brief overview of the key concepts will help explain the following workflow. Here we will focus on the basics of Boltzmann Machines and Quantum Annealing. Please refer to our educational platform for further fundamental concepts of Quantum Machine Learning and Quantum Computing.

## (Deep) Boltzmann Machines


Boltzmann Machines (BMs)  are undirected stochastic neural networks composed of $n$ units, which can be grouped into $n_v$ visible units $v$ and (optionally) $n_h$ hidden units $h$.
For supervised discriminative learning, the visible units $v$ include both the input features and the label. The input units are fixed to the observed data (e.g., pixel intensities) during training. Since they are always clamped, they do nothing more than providing biases to the rest of the units within the system (see $b^{\text{eff}}_i$). The hidden units are used to capture complex correlations in the input, capturing higher-order features. Consequently, the label units are used to read out the predicted class after the sampling process.
 These units, hidden and label units, can take on the values 0 or 1 and switch between these states with a certain probability. The probability of finding the system in a certain configuration is determined by the energy function of the system:

$$E(v, h) = -\sum_{i} b^{\text{eff}}_i s_i - \sum_{\substack{i,j\\i<j}} W_{i,j}s_is_j$$

with

$$b^{\text{eff}}_i = b_i + \sum_{k} W_{i,k}x_k.$$

Here, $s$ represents the state of hidden and label units, $b_i$ are the biases, and $W_{i,j}$ are the weights between units. Together, they define the energy of the system’s current configuration. Configurations with lower energy are more probable than those with higher energy, with their distribution governed by the well-known Boltzmann distribution. The connectivity can be limited to layers of hidden units to form a Deep Boltzmann machine (DBM).


Now, only the first layer of hidden units is directly connected to the input units, while the subsequent layers are only connected to the previous layer of hidden units.
 With this multilayer structure, the concept of hierarchy can be introduced, allowing the model to learn layers of increasingly abstract features in latent space.
 Additionally, we introduce the concept of convolution to the DBM. Weight sharing and local connectivity are the key features of convolutional layers, which allow the model to capture spatial hierarchies and local features in data. This is particularly beneficial for image classification tasks, as it enables the model to learn patterns that are invariant to translation and other transformations. By applying convolutional layers and hierarchical feature learning to the BM, we can create a Convolutional Deep Boltzmann Machine (CDBM) that is well-suited for image classification tasks, such as surface defect classification.
 The image below shows the structure of a CDBM.

<center>

<img src="notebook_src/images/figure_cdqbm.png" width="700">

</center>

The model introduced above is capable of capturing complex distributions of local and hierarchical features, as well as their relationship to the label. The next question is how such a model can be learned.

Training a Boltzmann Machine aims to adjust the weights and biases so that the model reproduces the dependencies observed in the training data for a given input. This means that when certain features are strongly associated with a particular label in the training set, the BM’s sampling distribution should reproduce that same distribution. BMs are therefore universal approximators of probability distributions. For this purpose the training process of supervised BMs consists of two separate phases: the clamped (also called positive) and the unclamped (negative) phase. In the clamped phase, the label units are additionally clamped to the true label of the current data point. In the unclamped phase the label units are set to evolve freely, while the input units still remain clamped. The difference between the average configuration of these two phases is then used to determine the weight updates.


#### Quantum Annealing and Quantum Boltzmann Machines

In order to train a BM or to use it for inference, samples from the Boltzmann distribution must be generated. This is often done using Markov Chain Monte Carlo methods like Gibbs sampling (Abändern -> more general description). However, these methods can be slow especially for large fully connected BMs, which is why this type of ML-model has been deemed impractical for many real-world applications in the past.

Quantum Annealing (QA) is a highly promising alternative approach for the sampling process. This quantum algorithm is designed to find minimum energy states of systems using effects such as quantum tunneling and superposition. Through these quantum phenomena, QA can potentially explore the energy landscape more efficiently than classical methods, especially in complex and high-dimensional spaces. The resulting sampling distribution, however, does not always exactly follow the Boltzmann distribution. This can be traced back to various factors, such as the physical temperature of the hardware, the freeze-out effect, and overall noise. To mitigate these effects, the parameters of the BM can be rescaled using an effective temperature parameter $\beta_{\text{eff}}$. Nonetheless, studies have shown that though the sample distribution may deviate from the true Boltzmann distribution, it can still be effectively used for training and inference in BMs as if it was the true distribution. Therefore, in this notebook, we will simply set $\beta_{\text{eff}}$ to 1.0 and, thus, skip this rescaling step.

With this framework, QA-based Quantum Boltzmann Machines (QBMs) leverage the capabilities of this algorithm to perform the sampling process required for training and inference potentially more efficiently than classical methods. The model, however, remains the same as in classical BMs.


## Setup & Imports




In [ ]:
import qbm as QBM
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import torch
from tqdm import tqdm

SEED = 42

## Sources

- [1] Review of Surface-Defect Detection Methods for Industrial Products Based on Machine Vision: https://doi.org/10.1109/ACCESS.2025.3571297
- [2] Prediction of Micro-Cracks in Steel Structures Subjected to Fatigue by Means of Acoustic Emission: https://doi.org/10.1007/s10921-025-01255-0
- [3] Permute-MAML: exploring industrial surface defect detection algorithms for few-shot learning: https://doi.org/10.1007/s40747-023-01219-9
- [4] Scaling Advantage in Approximate Optimization with Quantum Annealing: https://doi.org/10.1103/PhysRevLett.134.160601